# Frame_Extraction.ipynb

## Preprocessing (Frame-Sampling)

### Ziel
Alle `.mp4`-Videos aus den Datensätzen (z. B. `Top_100`) sollen durchsucht und Frames (Bilder) in einer festgelegten Rate extrahiert werden.  
Dadurch entsteht eine Bildbasis, die für weitere Analysen (z. B. Objekterkennung, Bewegungsanalyse) genutzt werden kann.

---

## Algorithmus / Tool
**OpenCV** (`cv2.VideoCapture`)

---

## Prozess

1. **Vorhandene Frames prüfen**  
   Scanne das `OUTPUT_DIR`, um eine Liste aller Videos zu erstellen, für die bereits Frames existieren.

2. **Videos laden**  
   Lese alle `.mp4`-Dateien aus dem Quellverzeichnis.

3. **Doppelte Verarbeitung vermeiden**  
   Wenn für ein Video bereits Frames vorhanden sind, überspringe es.

4. **Videoeigenschaften auslesen**  
   Öffne jedes neue Video mit `cv2.VideoCapture` und lies die FPS (Frames per Second) aus.

5. **Sampling-Rate bestimmen**  
   Berechne, **jedes wievielte Frame** gespeichert werden muss, um die gewünschte Extraktionsrate zu erreichen (z. B. 1 Frame pro Sekunde).

6. **Frames speichern**  
   Speichere die extrahierten Frames als `.jpg`-Dateien im Zielverzeichnis, z. B.: frames/<video_id>/frame_0001.jpg

## 1. Konfiguration
Hier werden die Pfade und die Sampling-Rate definiert. Diese Zelle muss als Erstes ausgeführt werden.

In [1]:
import cv2
import os
import math

# --- Konfiguration ---

# 1. Quellverzeichnis: Der Ordner, der die Video-Unterordner (Top_100, etc.) enthält
VIDEO_ROOT_DIR = "tiktok_100_vs_100"

# 2. Zielverzeichnis: Hier werden die extrahierten Frames gespeichert
OUTPUT_DIR = "data/processed/video_frames"

# 3. Sampling-Rate: Wie viele Frames pro Sekunde sollen gespeichert werden?
FRAMES_PER_SECOND_TO_SAVE = 1

# Beurteilung der Methodik

## Extraktion von 1 Frame pro Sekunde (FPS)

Die Wahl, **1 Frame pro Sekunde** zu extrahieren, stellt einen **kritischen Kompromiss** zwischen Datenmenge und Informationsgehalt dar.

---

### Vorteil
- **Reduktion der Datenmenge:**  
  Von etwa **30 Frames/Sekunde** auf **1 Frame/Sekunde** (Faktor 30).  
  Bei **200 Videos à 30 Sekunden** ergibt das:
  - ca. **6.000 Frames** statt **180.000 Frames**.  
- Diese Reduktion macht die nachfolgenden Analysen mit **YOLO** und **DeepFace** überhaupt erst **praktisch durchführbar**, da sie sonst zu rechenintensiv wären.

---

### Nachteil (Risiko)
- **Schnelle Ereignisse** (z. B. Szenenwechsel von 0,5 Sekunden) oder **kurze Texteinblendungen** können **übersehen** werden.  
- Die zeitliche Auflösung ist somit eingeschränkt, was zu **verpassten Details** führen kann.

---

### Erkenntnis (Speicher)
- Das erzeugte Ergebnis umfasst etwa **1–2 GB an JPEG-Dateien**.  
- Diese Datenmenge ist **signifikant** und sollte bei der **Projektübergabe und Datenspeicherung** berücksichtigt werden.

## 2. Implementierung der Extraktions-Funktion
Diese Zelle definiert die Hauptfunktion.

In [ ]:
def extract_frames():
    """
    Durchläuft alle .mp4-Dateien im VIDEO_ROOT_DIR, extrahiert 
    Frames basierend auf FRAMES_PER_SECOND_TO_SAVE und speichert sie im OUTPUT_DIR.
    
    ÜBERSPRINGT Videos, für die bereits Frames im OUTPUT_DIR existieren.
    """
    
    # 1. Sicherstellen, dass das Ausgabeverzeichnis existiert
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Ausgabeverzeichnis '{OUTPUT_DIR}' ist bereit.")
    
    # --- PRÜFUNG ---
    print("Analysiere existierende Frames, um Duplikate zu vermeiden...")
    # Baut ein Set aller Video-Basisnamen, die bereits Frames im Ordner haben.
    # z.B. {'top_43_likes_822300_id_7493024973506792746', ...}
    existing_video_basenames = set()
    try:
        for f in os.listdir(OUTPUT_DIR):
            if f.endswith('.jpg'):
                # Basisname ist alles vor "_frame_...jpg"
                base_name = "_".join(f.split("_")[:-2])
                existing_video_basenames.add(base_name)
    except FileNotFoundError:
        print(f"WARNUNG: {OUTPUT_DIR} noch nicht gefunden. Wird neu erstellt.")
        
    print(f"{len(existing_video_basenames)} Videos scheinen bereits verarbeitet worden zu sein.")
    # --- PRÜFUNG ---
    
    processed_count = 0
    skipped_count = 0
    
    # 2. Rekursiv durch alle Ordner und Dateien im Haupt-Videoordner laufen
    for subdir, dirs, files in os.walk(VIDEO_ROOT_DIR):
        for filename in files:
            if filename.endswith(".mp4"):
                
                video_path = os.path.join(subdir, filename)
                video_basename = os.path.splitext(filename)[0]

                if video_basename in existing_video_basenames:
                    skipped_count += 1
                    continue
                
                try:
                    # 3. Video mit OpenCV öffnen
                    cap = cv2.VideoCapture(video_path)
                    if not cap.isOpened():
                        print(f"FEHLER: Konnte Video nicht öffnen: {video_path}")
                        continue

                    # 4. FPS (Frames Pro Sekunde) des Videos auslesen
                    fps = cap.get(cv2.CAP_PROP_FPS)
                    if not fps or fps == 0:
                        fps = 30 # Fallback

                    # 5. Frame-Intervall berechnen
                    frame_skip_interval = math.floor(fps / FRAMES_PER_SECOND_TO_SAVE)
                    if frame_skip_interval < 1:
                        frame_skip_interval = 1 

                    print(f"\nVerarbeite NEUES Video: {filename} (FPS: {fps:.2f}, speichere jedes {frame_skip_interval}. Frame)")

                    frame_count = 0
                    saved_frame_count = 0

                    # 6. Schleife durch alle Frames im Video
                    while cap.isOpened():
                        ret, frame = cap.read()
                        if not ret:
                            break
                        
                        if frame_count % frame_skip_interval == 0:
                            output_filename = f"{video_basename}_frame_{frame_count}.jpg"
                            output_path = os.path.join(OUTPUT_DIR, output_filename)
                            cv2.imwrite(output_path, frame)
                            saved_frame_count += 1
                        
                        frame_count += 1

                    cap.release()
                    print(f"-> Fertig. {saved_frame_count} Frames gespeichert.")
                    processed_count += 1

                except Exception as e:
                    print(f"FEHLER bei der Verarbeitung von {video_path}: {e}")
                    if 'cap' in locals() and cap.isOpened():
                        cap.release()

    print(f"\n--- Verarbeitung abgeschlossen ---")
    print(f"Insgesamt {processed_count} NEUE Videos verarbeitet.")
    print(f"{skipped_count} Videos wurden übersprungen (bereits vorhanden).")
    print(f"Alle Frames sind in '{OUTPUT_DIR}'.")

## 3. Ausführung
Das Ausführen dieser Zelle startet den gesamten Prozess.

In [3]:
extract_frames()

Ausgabeverzeichnis 'data/processed/video_frames' ist bereit.
Analysiere existierende Frames, um Duplikate zu vermeiden...
197 Videos scheinen bereits verarbeitet worden zu sein.

--- Verarbeitung abgeschlossen ---
Insgesamt 0 NEUE Videos verarbeitet.
197 Videos wurden übersprungen (bereits vorhanden).
Alle Frames sind in 'data/processed/video_frames'.
